# Agentic Analysis: A Metrics Taxonomy

Wiki reference for [agentic analysis](https://ml-viz-ruby.vercel.app/wiki/agent-metrics-taxonomy).

**The idea in one sentence.** You measure an agentic system at **four levels** — one agent on one
task (correctness), one agent over many tasks (a profile), a *team* on one task (trajectory
correctness), and the whole deployment (system health) — and a metric that is diagnostic at one
level is misleading at another.

**Where it shows up.** Every production agent report: did the coding agent's patch pass the hidden
tests (Level 1)? what's its p95 steps-per-task (Level 2)? did the planner hand the coder the right
spec (Level 3)? what's our containment rate and cost-per-resolved-task (Level 4)?

In this notebook we **simulate a fleet of trajectories**, compute the metrics from scratch, **
cross-check them against a pandas one-liner**, visualise the latency/cost scaling that decides
usability, then cover tradeoffs and leave you a scaffold.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})
BRAND, TEAL, YELLOW, ROSE = '#6366f1', '#14b8a6', '#eab308', '#f43f5e'
rng = np.random.default_rng(7)

## 1 · A synthetic trajectory log

One agent, `TOOLS` available. A **trajectory** is a list of steps; each step is a
`(tool, args, ok)` triple. We also record the oracle **minimum steps** for the task and whether
the final outcome was **solved**. This is the raw material every metric is computed from.

In [ ]:
TOOLS = ['search', 'read_file', 'run_tests', 'edit', 'lookup_order']
TOOL_FAIL = {'search': 0.05, 'read_file': 0.03, 'run_tests': 0.02, 'edit': 0.08, 'lookup_order': 0.25}

def simulate_trajectory(task_id, difficulty):
    '''One attempt at one task. Harder tasks take more steps and sometimes loop.'''
    min_steps = 3 + difficulty                      # oracle lower bound
    # steps taken >= min_steps, inflated by wandering that scales with difficulty
    steps_taken = min_steps + rng.poisson(difficulty * 0.9)
    trace = []
    for _ in range(steps_taken):
        tool = TOOLS[rng.integers(len(TOOLS))]
        # low-cardinality args so accidental repeats (loops) are possible
        args = 'q' + str(rng.integers(4))
        ok = rng.random() > TOOL_FAIL[tool]
        trace.append((tool, args, ok))
    # solved with prob that falls as the trajectory bloats past the oracle
    p_solve = np.clip(0.95 - 0.12 * (steps_taken - min_steps), 0.15, 0.97)
    solved = rng.random() < p_solve
    return {'task_id': task_id, 'trace': trace, 'min_steps': min_steps, 'solved': solved}

N = 400
difficulties = rng.integers(1, 6, size=N)
log = [simulate_trajectory(i, int(difficulties[i])) for i in range(N)]
print('trajectories:', len(log))
print('example steps:', log[0]['trace'][:3], '... solved=', log[0]['solved'])

### Level 1 → Level 2: correctness and efficiency

**Success rate** is the mean of the solved flags. **Efficiency ratio** normalises steps against
the oracle: $\text{eff} = \text{min\_steps} / \text{steps\_taken} \in (0, 1]$. **Loop score**
counts repeated `(tool, args)` pairs inside a single trajectory — the signature of a reasoning
loop. Note we report the **p95** of steps, not the mean: agent pathology lives in the tail.

In [ ]:
def steps_taken(t):      return len(t['trace'])
def efficiency(t):       return t['min_steps'] / steps_taken(t)
def loop_score(t):
    pairs = [(tool, args) for tool, args, _ in t['trace']]
    counts = Counter(pairs)
    return sum(c - 1 for c in counts.values() if c > 1)   # extra repeats

success_rate = np.mean([t['solved'] for t in log])
eff_mean     = np.mean([efficiency(t) for t in log])
steps_arr    = np.array([steps_taken(t) for t in log])
loop_mean    = np.mean([loop_score(t) for t in log])

print(f'success rate      : {success_rate:.3f}')
print(f'efficiency ratio  : {eff_mean:.3f}  (1.0 = oracle-optimal)')
print(f'steps  p50 / p95  : {np.percentile(steps_arr,50):.0f} / {np.percentile(steps_arr,95):.0f}')
print(f'mean loop score   : {loop_mean:.2f}')

### Per-tool error rate (find the broken tool)

A fleet-wide error rate hides *which* tool is misbehaving. Bucket every call by tool and divide.

In [ ]:
def per_tool_error_rate(log):
    total = defaultdict(int); fail = defaultdict(int)
    for t in log:
        for tool, _, ok in t['trace']:
            total[tool] += 1
            if not ok: fail[tool] += 1
    return {tool: fail[tool] / total[tool] for tool in total}

err = per_tool_error_rate(log)
for tool, r in sorted(err.items(), key=lambda kv: -kv[1]):
    print(f'{tool:12s} {r:5.1%}')

### Reliability under stochasticity: pass@k and pass^k

Agents are stochastic, so one run is noise. Run each task `k` times: **pass@k** = solved in *at
least one* attempt (capability ceiling), **pass^k** = solved in *all* attempts (consistency). A
big gap means the agent is capable-but-flaky — exactly what a single run hides.

In [ ]:
def run_k(task_id, difficulty, k):
    return [simulate_trajectory(task_id, difficulty)['solved'] for _ in range(k)]

k = 5
attempts = [run_k(i, int(difficulties[i]), k) for i in range(N)]
pass_at_k = np.mean([any(a) for a in attempts])
pass_pow_k = np.mean([all(a) for a in attempts])
print(f'pass@{k} (>=1 solve): {pass_at_k:.3f}')
print(f'pass^{k} (all solve): {pass_pow_k:.3f}')
print(f'flakiness gap      : {pass_at_k - pass_pow_k:.3f}')

## 2 · The library way — and a cross-check

In production you would not hand-roll `defaultdict` loops — you'd flatten the log into a
step-level table and let **pandas** aggregate. We compute the per-tool error rate that way and
**assert it matches** the from-scratch result to the last decimal.

In [ ]:
import pandas as pd

rows = [
    {'task_id': t['task_id'], 'step': i, 'tool': tool, 'args': args, 'ok': ok}
    for t in log for i, (tool, args, ok) in enumerate(t['trace'])
]
df = pd.DataFrame(rows)

err_pandas = (1 - df.groupby('tool')['ok'].mean()).to_dict()
for tool in TOOLS:
    assert np.isclose(err_pandas[tool], err[tool]), tool
print('from-scratch and pandas per-tool error rates match ✓')
print()
print(df.groupby('tool')['ok'].agg(calls='count', error_rate=lambda s: 1 - s.mean()).round(3))

## 3 · Visualise the scaling that decides usability

Two curves govern whether an agent ships. **Cost** grows *super-linearly* in steps because each
step re-sends the growing transcript:

$$\text{tokens}(S) = \sum_{s=1}^{S}(c_0 + s\,\bar m) = S c_0 + \tfrac{S(S+1)}{2}\,\bar m$$

**Latency** grows only with the *critical-path depth* — the sequential LLM chain plus any tool
calls that block the next step. Firing **independent** tool calls asynchronously overlaps them
with the next LLM call, so they leave the critical path.

In [ ]:
c0, mbar = 1500, 350          # fixed context, tokens added per step
LLM_MS, TOOL_MS = 800, 300    # per-step reasoning + tool latency (ms)
S = np.arange(1, 21)

tokens = S * c0 + mbar * S * (S + 1) / 2
linear = (c0 + mbar) * S       # what a naive 'cost is linear in steps' guess predicts

# latency: LLM chain is always sequential; a fraction f of tool calls are independent
def latency_ms(S, f):
    llm = LLM_MS * S
    dep_tools = TOOL_MS * S * (1 - f)                       # block the next step
    indep_tools = max(0, TOOL_MS - LLM_MS) * S * f          # overlap the next LLM call
    return llm + dep_tools + indep_tools

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(S, tokens, color=YELLOW, marker='o', ms=3, label='actual (quadratic)')
ax1.plot(S, linear, color='#555', ls='--', label='naive linear guess')
ax1.set_title('Cost: tokens per task'); ax1.set_xlabel('steps S'); ax1.set_ylabel('tokens')
ax1.legend(); ax1.grid(alpha=.3)

ax2.plot(S, latency_ms(S, 0.0) / 1000, color=BRAND, marker='o', ms=3, label='all tools blocking')
ax2.plot(S, latency_ms(S, 0.5) / 1000, color=TEAL, marker='o', ms=3, label='50% independent (parallel)')
ax2.set_title('Latency: end-to-end seconds'); ax2.set_xlabel('steps S'); ax2.set_ylabel('seconds')
ax2.legend(); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

**What to notice.** Left: doubling the trajectory from 10 to 20 steps roughly **quadruples**
tokens — the naive linear line diverges badly, which is why long trajectories are a cost
*emergency*, not a rounding error. Right: parallelising the independent tool calls bends the
latency line down without changing a single token on the left — **parallelism buys latency,
never dollars.**

### The tail is the story

Level-4 dashboards that report *mean* steps-per-task bless a fleet that is quietly looping. Plot
the distribution and mark the mean vs p95.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(steps_arr, bins=range(steps_arr.min(), steps_arr.max() + 2), color=BRAND, alpha=.8)
ax.axvline(steps_arr.mean(), color=TEAL, ls='--', label=f'mean {steps_arr.mean():.1f}')
ax.axvline(np.percentile(steps_arr, 95), color=ROSE, ls='--', label=f'p95 {np.percentile(steps_arr,95):.0f}')
ax.set_title('Steps per task — agent failure lives in the tail')
ax.set_xlabel('steps'); ax.set_ylabel('trajectories'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 4 · Tradeoffs & when to use each level

| Level | Unit | Reach for it when… | Failure mode it catches |
|---|---|---|---|
| 1 · Single-agent correctness | one agent, one task | debugging a specific failure; regression tests | wrong tool, hallucinated observation, unsafe-but-successful path |
| 2 · Per-agent metrics | one agent, many tasks | comparing prompts/models; capacity planning | thrashing (p95 steps), a broken tool, cost blowups |
| 3 · Multi-agent trajectory | a team, one task | a hand-off pipeline underperforms its parts | lost context at a hand-off, redundant work, deadlock |
| 4 · System analysis | the deployment | product & ops health; go/no-go to ship | low containment, tail latency, unit-economics blowup |

**Leading vs lagging.** Success rate (and containment) are *lagging* — they drop after users are
hurt. Tool-selection accuracy, grounding, efficiency ratio, calibration, and (multi-agent)
routing accuracy + context-propagation fidelity are *leading*: they move first.

**Complexity.** All metrics here are $O(\text{steps})$ to compute per trajectory; the only cost
is the *plumbing* — you need per-step traces with a shared `trace_id`, not flat log lines.

**Common failure modes of the *measurement* itself:** averaging away the tail; scoring only the
final answer; trusting an un-validated LLM-judge on trajectory questions; and reporting one
wall-clock latency instead of splitting LLM vs tool time and critical-path depth.

## ✏️ Your turn

**Exercise — cost-per-success.** The single most decision-relevant Level-2 metric is not raw
accuracy but **cost per solved task**: total dollars spent divided by the number of tasks solved.
An agent that solves 60% at \$0.02 can beat one that solves 65% at \$0.40.

Fill in `cost_per_success` below. Use `tokens_for(steps)` for the token count of a trajectory and
`PRICE` per token; only *solved* trajectories count in the denominator, but *every* trajectory
costs money.

In [ ]:
PRICE = 5e-6   # $ per token (blended)
def tokens_for(steps):
    s = np.arange(1, steps + 1)
    return (c0 + s * mbar).sum()

def cost_per_success(log):
    # TODO(you): total $ spent across ALL trajectories, divided by the number SOLVED.
    total_cost = ...        # sum over log of tokens_for(steps_taken(t)) * PRICE
    n_solved = ...          # count of t['solved']
    return total_cost / n_solved

# cps = cost_per_success(log)
# print(f'cost per solved task: ${cps:.4f}')

In [ ]:
# --- assertion cell: passes silently when your implementation is correct ---
def _ref_cost_per_success(log):
    total = sum(tokens_for(steps_taken(t)) * PRICE for t in log)
    solved = sum(1 for t in log if t['solved'])
    return total / solved

assert np.isclose(cost_per_success(log), _ref_cost_per_success(log)), 'not matching the reference yet'
print(f'✓ cost per solved task: ${cost_per_success(log):.4f}')

<details>
<summary>Solution</summary>

```python
def cost_per_success(log):
    total_cost = sum(tokens_for(steps_taken(t)) * PRICE for t in log)
    n_solved = sum(1 for t in log if t['solved'])
    return total_cost / n_solved
```

The numerator sums over *every* trajectory (failures cost money too); the denominator counts
only solves. That asymmetry is the whole point — a flaky agent that retries a lot has a great
success rate and a terrible cost-per-success.
</details>

**Stretch.** Add a `containment_rate` (Level 4): mark a trajectory as *escalated* when its
`loop_score > 2` **or** `steps_taken > p95`, then compute the share that were *not* escalated.
That's the fraction a human never had to touch — the headline metric for a support agent.

## 6 · Key takeaways

- **Four levels, four questions:** one-task correctness → per-agent profile → team trajectory
  correctness → system health. A metric diagnostic at one level misleads at another.
- **Report tails, not means** — p95 steps-per-task and per-tool error rates carry the signal.
- **Latency follows critical-path depth; cost grows quadratically in steps.** Parallelise
  independent tool calls to cut latency; the tokens (and dollars) don't move.
- **Watch leading indicators** (tool-selection accuracy, grounding, efficiency, calibration,
  routing, context-propagation fidelity) — they drop *before* success rate does.
- **Cost-per-success and containment rate** are the numbers that actually decide whether an agent
  ships.

**Next:** the [Evaluating Agents lesson](https://ml-viz-ruby.vercel.app/courses/agent-design-patterns/10-evaluating-agents)
and the [Agent Observability wiki](https://ml-viz-ruby.vercel.app/wiki/agent-observability) that
produces these numbers in production.